# Milestone 1: FMO Energy Transfer — Haken-Ströbl Model

**Goal:** Simulate quantum energy transfer in the Fenna-Matthews-Olson (FMO) complex using the Haken-Ströbl model in QuTiP.

This reproduces the classical quantum simulation from:
> Guimarães et al. (2020) *Simulation of non-radiative energy transfer in photosynthetic systems using a quantum computer* — [arXiv:2009.01283](https://arxiv.org/abs/2009.01283)

**The FMO complex** is a pigment-protein complex in green sulfur bacteria with 7 coupled bacteriochlorophyll (BChl) molecules. It acts as a quantum wire between the antenna (chlorosome) and the reaction centre.

We model it as a network of coupled two-level systems with dephasing noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt

print(f'QuTiP version: {qt.__version__}')

## 1. Define the FMO Hamiltonian

The site energies and couplings below are taken from published FMO parameter sets (Adolphs & Renger 2006, widely used in the literature).

Units: wavenumbers (cm⁻¹). We'll work in units where ħ = 1.

In [ ]:
# FMO Hamiltonian (7-site, cm^-1, Adolphs & Renger 2006)
# Site energies on diagonal, couplings off-diagonal
H_fmo = np.array([
    [12410,  -87.7,   5.5,  -5.9,   6.7,  -13.7,  -9.9],
    [ -87.7, 12530,  30.8,   8.2,   0.7,   11.8,   4.3],
    [   5.5,  30.8, 12210, -53.5,  -2.2,   -9.6,   6.0],
    [  -5.9,   8.2, -53.5, 12320,  -70.7,  -17.0, -63.3],
    [   6.7,   0.7,  -2.2, -70.7,  12480,   81.1,  -1.3],
    [ -13.7,  11.8,  -9.6, -17.0,   81.1,  12630,  39.7],
    [  -9.9,   4.3,   6.0, -63.3,   -1.3,   39.7, 12440]
], dtype=float)

# Shift to zero baseline for numerical stability
H_fmo -= np.eye(7) * H_fmo[0, 0]

# Convert to QuTiP Qobj
H = qt.Qobj(H_fmo)
print('FMO Hamiltonian (7x7):')
print(H)

## 2. Haken-Ströbl Dephasing

The Haken-Ströbl model adds dephasing (pure decoherence) via collapse operators.
Each site interacts independently with the phonon bath.

Dephasing rate γ controls the noise level — we'll sweep this to see its effect on transport.

In [ ]:
def make_collapse_ops(n_sites, gamma):
    """Dephasing collapse operators for Haken-Ströbl model."""
    c_ops = []
    for i in range(n_sites):
        op = qt.basis(n_sites, i) * qt.basis(n_sites, i).dag()
        c_ops.append(np.sqrt(gamma) * op)
    return c_ops

# Parameters
n_sites = 7
gamma = 100.0  # dephasing rate (cm^-1) — typical room temperature value
t_end = 0.5    # ps
n_steps = 500
tlist = np.linspace(0, t_end, n_steps)

# Initial state: excitation on site 1 (BChl 1, input from antenna)
psi0 = qt.basis(n_sites, 0)  # site index 0 = BChl 1
rho0 = psi0 * psi0.dag()

c_ops = make_collapse_ops(n_sites, gamma)
print(f'Collapse operators: {len(c_ops)}')
print(f'Initial state: excitation on site 1')

## 3. Time Evolution — Lindblad Master Equation

In [ ]:
# Solve the Lindblad master equation
result = qt.mesolve(H, rho0, tlist, c_ops, [])

# Extract site populations over time
populations = np.zeros((len(tlist), n_sites))
for t_idx, rho in enumerate(result.states):
    for site in range(n_sites):
        populations[t_idx, site] = np.real(rho[site, site])

print(f'Simulation complete. Shape: {populations.shape}')

## 4. Plot Energy Transfer Dynamics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.cm.viridis(np.linspace(0, 1, n_sites))

for site in range(n_sites):
    ax.plot(tlist, populations[:, site], 
            label=f'BChl {site+1}', 
            color=colors[site],
            linewidth=2)

ax.set_xlabel('Time (ps)', fontsize=13)
ax.set_ylabel('Population', fontsize=13)
ax.set_title(f'FMO Energy Transfer — Haken-Ströbl Model (γ = {gamma} cm⁻¹)', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xlim(0, t_end)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/fmo_haken_strobl_populations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to results/figures/')

## 5. Experiment: Sweep Dephasing Rate

Key question from HYP-001: How does coherence lifetime change with temperature/dephasing?

Here we sweep γ from low noise (quantum limit) to high noise (classical limit) and observe how this affects transfer efficiency to the reaction-centre-adjacent site (BChl 3 or 4).

In [ ]:
gamma_values = [10, 50, 100, 300, 1000]  # cm^-1
target_site = 3  # BChl 4 — closest to reaction centre in FMO
t_probe = 0.3    # ps — probe efficiency at this time

results_sweep = {}

for g in gamma_values:
    c_ops_g = make_collapse_ops(n_sites, g)
    res = qt.mesolve(H, rho0, tlist, c_ops_g, [])
    # Population at target site at t_probe
    t_idx = np.argmin(np.abs(tlist - t_probe))
    pop_target = np.real(res.states[t_idx][target_site, target_site])
    results_sweep[g] = pop_target
    print(f'γ = {g:5d} cm⁻¹ → P(BChl {target_site+1}, t={t_probe}ps) = {pop_target:.4f}')

# Plot sweep
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(list(results_sweep.keys()), list(results_sweep.values()), 
            'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('Dephasing rate γ (cm⁻¹)', fontsize=13)
ax.set_ylabel(f'Population at BChl {target_site+1} (t={t_probe}ps)', fontsize=12)
ax.set_title('Environment-Assisted Quantum Transport in FMO', fontsize=14)
ax.grid(alpha=0.3)
ax.axvline(x=100, color='orange', linestyle='--', label='Room temperature (~300K)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/fmo_dephasing_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## Next Steps

- [ ] Compare output with Figure 3 of Guimarães et al. 2020
- [ ] Encode this system as a quantum circuit (→ Milestone 2)
- [ ] Add a sink term at BChl 3 to model irreversible trapping at reaction centre
- [ ] Log findings in `docs/hypothesis_log.md` (HYP-001)

---
*Notebook: 01_fmo_haken_strobl.ipynb | Last updated: May 2026*